# 📘 표본 통계량의 성질

**표본 통계량**(sample statistic)은 표본에서 계산한 값입니다.
같은 모집단에서 표본을 여러 번 추출하면, 표본평균이나 표본분산이
매번 다른 값을 가지게 됩니다.

이 노트북에서는 **표본평균이 어떤 성질을 가지는지** 시뮬레이션으로 확인합니다.

**학습 목표:**
- 표본평균의 분포와 대수법칙
- 표본 크기에 따른 표본평균의 수렴
- 표본평균의 표준편차는 모표준편차보다 작다
- 표준오차(Standard Error)의 개념
- 표본분산과 불편분산의 차이 (시뮬레이션으로 확인)
- 중심극한정리의 직관적 이해

## 1. 준비 — 모집단 정의

시뮬레이션을 위해 평균 μ=4, 표준편차 σ=0.8인 **정규분포 모집단**을 사용합니다.
이것은 "호수의 물고기 길이가 평균 4cm, 표준편차 0.8cm를 따른다"고 가정하는 것과 같습니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  라이브러리 임포트                          │
# │  numpy → 수치 계산                          │
# │  scipy.stats → 통계 분포                    │
# │  matplotlib, seaborn → 그래프               │
# └───────────────────────────────────────────┘

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# 스타일 설정
sns.set()
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 모집단: 정규분포 N(평균=4, 표준편차=0.8)
population = stats.norm(loc=4, scale=0.8)

# 모집단 통계량 확인
print("=== 모집단 (정규분포) ===")
print(f"모평균 μ: 4.0")
print(f"모표준편차 σ: 0.8")
print(f"모분산 σ²: 0.64")

# 모집단에서 표본 5개 추출해 보기
np.random.seed(1)
sample_5 = population.rvs(size=5)
print(f"\n표본 5개: {np.round(sample_5, 4)}")
print(f"표본평균: {np.mean(sample_5):.4f}")

## 2. 표본평균을 여러 번 계산하기

**표본평균**은 표본을 추출할 때마다 다른 값을 가직니다.
같은 모집단에서 10,000번 표본을 추출하여 표본평균의 분포를 확인합니다.

> 💡 중요한 점: 표본평균의 **평균**은 모평균에 가긌고,
> 표본평균의 **표준편차**는 모표준편차보다 작습니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본평균을 10,000번 계산                    │
# │  "10개 데이터 추출 → 평균 계산"을 10,000회 반복│
# │  표본평균의 분포를 확인할 수 있음            │
# └───────────────────────────────────────────┘

n_trial = 10000
sample_mean_array = np.zeros(n_trial)

np.random.seed(1)
for i in range(n_trial):
    sample = population.rvs(size=10)
    sample_mean_array[i] = np.mean(sample)

print(f"시뮤레이션 횟수: {n_trial}")
print(f"표본 크기: 10")
print(f"\n표본평균들의 평균: {np.mean(sample_mean_array):.4f}")
print(f"모평균 μ: 4.0000")
print(f"\n→ 표본평균의 평균이 모평균에 매우 가까움!")

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본평균의 분포 시각화                     │
# │  표본평균도 하나의 확률변수                 │
# │  모평균을 중심으로 종 모양 분포              │
# └───────────────────────────────────────────┘

se = np.std(sample_mean_array, ddof=1)

plt.figure(figsize=(8, 5))
sns.histplot(sample_mean_array, bins=30, color='steelblue',
             edgecolor='white', kde=True)
plt.axvline(np.mean(sample_mean_array), color='red', linestyle='--',
            label=f'표본평균의 평균={np.mean(sample_mean_array):.3f}')
plt.axvline(4.0, color='green', linestyle=':',
            label=f'모평균 μ=4.0')
plt.title("표본평균의 분포 (10,000회 시뮮레이션, n=10)")
plt.xlabel("표본평균")
plt.ylabel("빈도")
plt.legend()
plt.tight_layout()
plt.savefig("sample_mean_dist.png", dpi=100)
plt.show()

print(f"표본평균의 평균: {np.mean(sample_mean_array):.4f}")
print(f"표본평균의 표준편차: {se:.4f}")
print(f"모표준편차/√n: {0.8/np.sqrt(10):.4f}")
print(f"\n→ 표본평균의 표준편차 ≈ 모표준편차/√n")

## 3. 대수법칙 — 표본 크기가 크면 표본평균이 모평균에 수렴

**대수법칙**(Law of Large Numbers): 표본 크기 n이 커지면
표본평균이 모평균 μ에 가까워집니다.

> 💡 직관적으로 이해하면: 데이터를 더 많이 모으려면,
> 계산된 평균이 "진짜 평균"에 더 가까워집니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본 크기에 따른 표본평균의 변화            │
# │  표본 크기를 10→100,000으로 증간            │
# │  표본평균이 모평균(4.0)에 수렴하는지 확인    │
# └───────────────────────────────────────────┘

size_array = np.arange(start=10, stop=100100, step=100)
sample_mean_array_size = np.zeros(len(size_array))

np.random.seed(1)
for i in range(len(size_array)):
    sample = population.rvs(size=size_array[i])
    sample_mean_array_size[i] = np.mean(sample)

plt.figure(figsize=(8, 5))
plt.plot(size_array, sample_mean_array_size, color='steelblue', alpha=0.7)
plt.axhline(4.0, color='red', linestyle='--', label='모평균 μ=4.0')
plt.xlabel("표본 크기 (n)")
plt.ylabel("표본평균")
plt.title("대수법칙: 표본 크기가 클수록 표본평균이 모평균에 수렴")
plt.legend()
plt.tight_layout()
plt.savefig("law_of_large_numbers.png", dpi=100)
plt.show()
print("대수법칙 그래프 저장 완료")

## 4. 표본평균 계산 함수와 표본 크기별 분포 비교

표본평균을 여러 번 계산하는 함수를 만들어,
**표본 크기에 따른 표본평균의 분포 변화**를 비교합니다.

> 💡 표본 크기가 커지면:
> - 표본평균의 **평균**은 모평균에 가까워지고
> - 표본평균의 **퍼짐**(표준편차)은 작아집니다

In [ ]:
def calc_sample_mean(size, n_trial):
    """모집단에서 size개를 추출해 평균을 구하는 것을 n_trial번 반복"""
    sample_mean_array = np.zeros(n_trial)
    for i in range(n_trial):
        sample = population.rvs(size=size)
        sample_mean_array[i] = np.mean(sample)
    return sample_mean_array

np.random.seed(1)
result = calc_sample_mean(size=10, n_trial=10000)
print(f"표본 크기=10, 10,000회 시뮮레이션")
print(f"표본평균의 평균: {np.mean(result):.4f}")
print(f"표본평균의 표준편차: {np.std(result, ddof=1):.4f}")
print(f"모평균: 4.0000")

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본 크기에 따른 표본평균 분포 비교         │
# │  n=10, n=20, n=50에서 바이올린 플롣 비교    │
# │  표본 크기가 클수록 분포가 좁아짐            │
# └───────────────────────────────────────────┘

np.random.seed(1)
size_10 = calc_sample_mean(size=10, n_trial=10000)
size_20 = calc_sample_mean(size=20, n_trial=10000)
size_50 = calc_sample_mean(size=50, n_trial=10000)

sim_result = pd.DataFrame({
    "sample_mean": np.concatenate([size_10, size_20, size_50]),
    "size": ["n=10"] * 10000 + ["n=20"] * 10000 + ["n=50"] * 10000
})

plt.figure(figsize=(8, 5))
sns.violinplot(x="size", y="sample_mean", data=sim_result,
               order=["n=10", "n=20", "n=50"], color='lightgray')
plt.axhline(4.0, color='red', linestyle='--', label='모평균 μ=4.0')
plt.title("표본 크기에 따른 표본평균의 분포")
plt.xlabel("표본 크기")
plt.ylabel("표본평균")
plt.legend()
plt.tight_layout()
plt.savefig("sample_mean_violin.png", dpi=100)
plt.show()

print("표본 크기별 표본평균의 표준편차:")
print(f"  n=10: {np.std(size_10, ddof=1):.4f}")
print(f"  n=20: {np.std(size_20, ddof=1):.4f}")
print(f"  n=50: {np.std(size_50, ddof=1):.4f}")
print(f"\n→ 표본 크기가 클수록 표본평균의 퍼짐이 작아짐")

## 5. 표본평균의 표준편차는 모표준편차보다 작다

위 비우린 플롣에서 본 것과 같이, 표본평균의 표준편차는 모집단의 표준편차보다 작습니다.

이는 중요한 성질입니다:
- 모표준편차 σ = 0.8
- 표본평균의 표준편차는 σ/√n 이하
- 표본 크기 n이 커질수록 표본평균이 더 져중스러워짐

시뮮레이션으로 확인해 봅시다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본평균의 표준편차는 모표준편차보다 작다  │
# │  모표준편차 σ=0.8, 표본크기 n에 따른 변화  │
# │  표본평균의 표준편차는 n이 커질수록 감소   │
# └───────────────────────────────────────────┘

# 표본 크기별 표본평균의 표준편차
print("=== 표본 크기별 표본평균의 표준편차 ===")
for n in [2, 5, 10, 20, 50, 100]:
    np.random.seed(1)
    sm = calc_sample_mean(size=n, n_trial=10000)
    se = np.std(sm, ddof=1)
    print(f"  n={n:3d}: SE(시뮮레이션)={se:.4f}, SE(이론)={0.8/np.sqrt(n):.4f}")

print(f"\n모표준편차 σ: 0.8")
print(f"→ 모든 표본 크기에서 표본평균의 표준편차가 모표준편차보다 작음")

## 6. 표준오차 (Standard Error)

**표준오차**(SE)는 표본평균의 표준편차입니다.
이론적으로 다음 공식으로 계산됩니다:

$$SE = \frac{\sigma}{\sqrt{n}}$$

- σ: 모집단 표준편차
- n: 표본 크기

> 💡 표준오차가 작을수록 표본평균이 모평균에 가까이 있다는 의미입니다.
> 표본 크기 n이 커지면 SE가 작아집니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본평균의 표준편차 vs 표준오차             │
# │  시뮮레이션 결과 ≈ 이론값(σ/√n) 비굴       │
# │  표본 크기가 클수록 표준오차가 작아짐         │
# └───────────────────────────────────────────┘

size_array = np.arange(start=2, stop=102, step=2)
sample_mean_std_array = np.zeros(len(size_array))

np.random.seed(1)
for i in range(len(size_array)):
    sample_mean = calc_sample_mean(size=size_array[i], n_trial=10000)
    sample_mean_std_array[i] = np.std(sample_mean, ddof=1)

standard_error = 0.8 / np.sqrt(size_array)

plt.figure(figsize=(8, 5))
plt.plot(size_array, sample_mean_std_array, color='steelblue',
         linewidth=2, label='시뮮레이션 결과')
plt.plot(size_array, standard_error, color='red', linestyle='dotted',
         linewidth=2, label='이론값 SE = σ/√n')
plt.xlabel("표본 크기 (n)")
plt.ylabel("표본평균의 표준편차")
plt.title("표준오차: 시뮮레이션 vs 이론값")
plt.legend()
plt.tight_layout()
plt.savefig("standard_error.png", dpi=100)
plt.show()

print("표준오차 비교 (표본 크기 n=10):")
print(f"  시뮮레이션: {sample_mean_std_array[4]:.4f}")
print(f"  이론값 SE:  {standard_error[4]:.4f}")
print(f"  → 시뮮레이션 결과가 이론값에 매우 가까움")

## 7. 표본분산 vs 불편분산 — 편향 확인

1변량 데이터에서 배운 것과 같이, **표본분산**(N으로 나눬움)은
모분산보다 약간 작게 추정되는 **편향**이 있습니다.

**불편분산**(N-1로 나눬움)은 이 편향을 보정합니다.
시뮮레이션으로 직접 확인합니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본분산의 편향 확인                       │
# │  표본분산(N으로 나눬움)은 모분산보다 작음      │
# │  불편분산(N-1로 나눬움)은 모분산에 가까움       │
# │  10,000회 시뮮레이션으로 확인               │
# └───────────────────────────────────────────┘

n_trial = 10000

sample_var_array = np.zeros(n_trial)
unbias_var_array = np.zeros(n_trial)

np.random.seed(1)
for i in range(n_trial):
    sample = population.rvs(size=10)
    sample_var_array[i] = np.var(sample, ddof=0)   # 표본분산 (N)
    unbias_var_array[i] = np.var(sample, ddof=1)  # 불편분산 (N-1)

print("=== 모분산 vs 표본분산 vs 불편분산 ===")
print(f"모분산 σ²:           0.6400")
print(f"표본분산의 평균:     {np.mean(sample_var_array):.4f}  ← 모분산보다 작음 (편향)")
print(f"불편분산의 평균:     {np.mean(unbias_var_array):.4f}  ← 모분산에 가까움 (불편)")
print(f"\n→ 불편분산(ddof=1)이 모분산의 더 정확한 추정치임")

In [ ]:
# ├───────────────────────────────────────────┤
# │  표본 크기에 따른 불편분산 수렴              │
# │  표본 크기가 커지면 불편분산이 모분산에 수렴  │
# └───────────────────────────────────────────┘

size_array = np.arange(start=10, stop=100100, step=100)
unbias_var_array_size = np.zeros(len(size_array))

np.random.seed(1)
for i in range(len(size_array)):
    sample = population.rvs(size=size_array[i])
    unbias_var_array_size[i] = np.var(sample, ddof=1)

plt.figure(figsize=(8, 5))
plt.plot(size_array, unbias_var_array_size, color='steelblue', alpha=0.7)
plt.axhline(0.64, color='red', linestyle='--', label='모분산 σ²=0.64')
plt.xlabel("표본 크기 (n)")
plt.ylabel("불편분산")
plt.title("불편분산의 수렴: 표본 크기가 크면 모분산에 수렴")
plt.legend()
plt.tight_layout()
plt.savefig("unbiased_variance.png", dpi=100)
plt.show()
print("불편분산 수렴 그래프 저장 완료")

## 8. 중심극한정리 (Central Limit Theorem)

**중심극한정리**(CLT): 모집단 분포가 어떤 형태든 상관없이,
표본 크기 n이 충분히 크면 **표본평균의 분포**가 정규분포에 가까워집니다.

> 💡 이것은 통계학에서 가장 중요한 정리 중 하나입니다.
> 모집단이 정규분포가 아니어도, 표본평균은 정규분포를 따르는 경향이 있습니다!

동전 던지기(베르누이 분포)로 중심극한정리를 확인합니다.

In [ ]:
# ├───────────────────────────────────────────┤
# │  중심극한정리 시뮮레이션                     │
# │  동전 던지기 (앞=1, 뒤=0)를 n번 반복        │
# │  "앞이 나온 비율(=표본평균)"의 분포 확인     │
# │  모집단은 정규분포가 아님 (베르누이 분포!)   │
# │  그런데 표본평균의 분포는 정규분포에 근사  │
# └───────────────────────────────────────────┘

n_size = 10000
n_trial = 50000
coin = np.array([0, 1])
count_coin = np.zeros(n_trial)

np.random.seed(1)
for i in range(n_trial):
    result = np.random.choice(coin, size=n_size)
    count_coin[i] = np.mean(result)

plt.figure(figsize=(8, 5))
sns.histplot(count_coin, bins=50, color='steelblue',
             edgecolor='white', kde=True)
plt.axvline(0.5, color='red', linestyle='--', label='이론적 평균=0.5')
plt.title(f"동전 던지기: 앞이 나온 비율의 분포 (n={n_size:,}, {n_trial:,}회)")
plt.xlabel("앞이 나온 비율")
plt.ylabel("빈도")
plt.legend()
plt.tight_layout()
plt.savefig("clt_coin.png", dpi=100)
plt.show()

print(f"동전 던지기 {n_size:,}번을 {n_trial:,}회 반복")
print(f"평균: {np.mean(count_coin):.6f} (이론값: 0.5)")
print(f"표준편차: {np.std(count_coin, ddof=1):.6f}")
print(f"이론적 SE: {0.5/np.sqrt(n_size):.6f}")
print(f"\n→ 비정규 모집단(동전)에서도 표본평균이 정규분포를 따름!")

## 🎯 연습 문제

1. 표본 크기를 5, 20, 100으로 변경하여 각각 10,000회 시뮮레이션을 하고, 표본평균의 분포를 비교하세요.
2. 표준오차 공식(SE = σ/√n)을 사용하여 n=5, 10, 50, 100일 때의 SE를 계산하고, 시뮮레이션 결과와 비교하세요.
3. 표본분산과 불편분산의 평균을 각각 10,000회 시뮤레이션으로 계산하고, 다음 모분산(σ²=0.64)와 비교하세요.
4. 중심극한정리를 확인하기 위해, 다른 비정규 분포(예: 균등분포)에서도 표본평균의 분포가 정규분포 모양인지 확인하세요.
5. 표본 크기가 2배가 되면 표준오차는 얼마나 되는지 계산하세요.